# 05 - Preprocessing rieng cho Deep Learning

Notebook nay xu ly truc tiep tu `dataset/raw/New_Attack_Dataset.csv` de tao dataset phu hop hon cho Transformer/SecureBERT/ModernBERT.

Khac voi pipeline TF-IDF cu:
- Tao cot `DL_Text` lam input chinh cho deep learning.
- Giu ngu canh tu nhien, cau van, va dau cau co ich thay vi bien moi thu thanh chuoi token lowercase.
- Chuan hoa IOC/entity thanh special tokens than thien voi Transformer: `[URL]`, `[IPV4]`, `[HASH]`, `[WINDOWS_PATH]`, ...
- Van giu logic nhan quan trong tu pipeline cu: normalize label ve parent technique, rare-label mapping, merge duplicate text, split Stage 1/Stage 2.
- Xuat `special_tokens.json` de notebook train co the add tokens vao tokenizer.

In [ ]:
from pathlib import Path
import html
import json
import re
import sys
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from preprocessing import (  # reuse label/domain decisions from old pipeline
    RARE_LABEL_MAPPING,
    cti_tokenizer,
    get_label_frequency,
    is_valid_label_string,
    label_set,
    manual_resolve_hard_conflict,
    merge_label_strings,
    normalize_cti_text,
    normalize_labels,
    split_stage_datasets,
    update_rare_labels,
)

RAW_PATH = PROJECT_ROOT / "dataset" / "raw" / "New_Attack_Dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "dataset" / "processed_deeplearning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGE1_THRESHOLD = 30
MIN_WORDS_KEEP = 2
MAX_LABELS_AFTER_MAPPING = 3

print("Project root:", PROJECT_ROOT)
print("Raw path:", RAW_PATH)
print("Output dir:", OUTPUT_DIR)

## 1. Deep-learning text policy

`DL_Text` khong nen la `Tokenized_Text` lowercase vi Transformer can ngu canh tu nhien. Tuy nhien CTI co nhieu IOC/code/path ma tokenizer generic xu ly kem, nen ta:

- Giu casing va cau truc cau o muc vua phai.
- Trung hoa dinh danh cu the de tranh hoc vet: URL/IP/email/hash/CVE/ATT&CK ID.
- Giu tin hieu loai entity va keyword quan trong: `system32`, `startup folder`, `run key`, basename file co extension.
- Them signal tokens cho payload SQLi thay vi xoa het payload.
- Chi xoa boilerplate/step heading ro rang, khong lam mat hanh vi tan cong.

In [ ]:
SPECIAL_TOKENS = [
    "[MITRE_ID]", "[MITRE_REF]",
    "[URL]", "[EMAIL]", "[IPV4]", "[IPV6]", "[HASH]",
    "[CVE]", "[WINDOWS_PATH]", "[UNC_PATH]", "[UNIX_PATH]", "[REGISTRY_KEY]",
    "[SQLI_CONTEXT]", "[SQLI_BOOLEAN_TRUE]", "[SQLI_BOOLEAN_FALSE]", "[SQLI_COMMENT]",
    "[SQLI_TIME_DELAY]", "[SQLI_UNION_SELECT]", "[SQLI_ENUMERATION_FUNC]", "[SQLI_DB_VERSION]",
    "[BASE64]", "[POWERSHELL]", "[CMD]",
]

BOILERPLATE_PATTERNS = [
    r"\bAI Agents\s*&\s*LLM Exploits\b",
    r"\bAI/ML Security\b",
    r"\bRed Team\b",
    r"\bBlue Team\b",
    r"\bBeginner Friendly\b",
    r"\bOptional\b",
    r"\bSetup Lab\b",
    r"\bLab Setup\b",
    r"\bYou now have\b",
    r"\bGreat for demos\b",
    r"\bGreat for dataset generation\b",
]

def compact_spaces(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    text = re.sub(r"\s+([,.;:)])", r"\1", text)
    text = re.sub(r"([(])\s+", r"\1", text)
    return text.strip(" .,-;:")

def remove_light_boilerplate(text):
    text = str(text)
    text = re.sub(r"(?<=[a-z)])(?=(?:Step|Phase)\s+\d+\b)", ". ", text)
    text = re.sub(r"(?m)(^|\s)\d{1,2}\.\s+", " ", text)
    text = re.sub(r"\b(step|phase)\s+\d+\s*[:\-–—]?", " ", text, flags=re.IGNORECASE)
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern + r"\s*[.:]?", " ", text, flags=re.IGNORECASE)
    return compact_spaces(text)

def add_sqli_special_signals(text):
    text_str = str(text)
    lower_text = text_str.lower()
    tokens = []
    has_sqli_context = bool(re.search(
        r"(?:\b(?:sql injection|sqli|blind sql|time[- ]based sql|sqlmap|sqlninja|"
        r"union\s+(?:all\s+)?select|information_schema|waitfor\s+delay|pg_sleep)\b|"
        r"@@version|\bselect\b.{0,120}\bfrom\b)",
        lower_text,
        flags=re.IGNORECASE,
    ))
    if has_sqli_context:
        tokens.append("[SQLI_CONTEXT]")
    if re.search(r"\b(or|and)\s+['\"]?1['\"]?\s*=\s*['\"]?1['\"]?(?!\w)", text_str, flags=re.IGNORECASE):
        tokens.append("[SQLI_BOOLEAN_TRUE]")
    if re.search(r"\b(or|and)\s+['\"]?1['\"]?\s*=\s*['\"]?2['\"]?(?!\w)", text_str, flags=re.IGNORECASE):
        tokens.append("[SQLI_BOOLEAN_FALSE]")
    if has_sqli_context and re.search(r"(--|/\*|\*/)", text_str):
        tokens.append("[SQLI_COMMENT]")
    if has_sqli_context and re.search(r"\b(sleep|benchmark|pg_sleep|waitfor\s+delay)\b", text_str, flags=re.IGNORECASE):
        tokens.append("[SQLI_TIME_DELAY]")
    if re.search(r"\bunion\s+(all\s+)?select\b", text_str, flags=re.IGNORECASE):
        tokens.append("[SQLI_UNION_SELECT]")
    if has_sqli_context and re.search(r"\b(substring|substr|ascii|length|database|version|user|schema_name)\s*\(", text_str, flags=re.IGNORECASE):
        tokens.append("[SQLI_ENUMERATION_FUNC]")
    if re.search(r"@@version", text_str, flags=re.IGNORECASE):
        tokens.append("[SQLI_DB_VERSION]")
    if tokens:
        text_str = text_str + " " + " ".join(dict.fromkeys(tokens))
    return text_str

def normalize_cve_dl(text):
    def repl(match):
        year = match.group(0).split("-")[1]
        return f" [CVE] CVE_YEAR_{year} "
    return re.sub(r"\bCVE-\d{4}-\d{4,7}\b", repl, str(text), flags=re.IGNORECASE)

def normalize_paths_dl(text):
    text = str(text)

    def registry_repl(match):
        key = match.group(0).lower()
        hints = ["[REGISTRY_KEY]"]
        if "\\run" in key or "\\runonce" in key:
            hints.append("run key")
        if "services" in key:
            hints.append("services key")
        if "currentversion" in key:
            hints.append("currentversion")
        return " " + " ".join(dict.fromkeys(hints)) + " "

    def windows_repl(match):
        path = match.group(0)
        lower = path.lower()
        hints = ["[WINDOWS_PATH]"]
        for keyword in ["system32", "syswow64", "appdata", "programdata", "startup", "temp"]:
            if keyword in lower:
                hints.append(keyword)
        basename = re.split(r"[\\/]", path)[-1].strip().lower()
        if re.search(r"\.[a-z0-9]{1,6}$", basename):
            hints.append(basename)
        return " " + " ".join(dict.fromkeys(hints)) + " "

    def unc_repl(match):
        path = match.group(0)
        basename = re.split(r"[\\/]", path)[-1].strip().lower()
        hints = ["[UNC_PATH]", "network share"]
        if re.search(r"\.[a-z0-9]{1,6}$", basename):
            hints.append(basename)
        return " " + " ".join(dict.fromkeys(hints)) + " "

    text = re.sub(r"\b(HKLM|HKCU|HKEY_LOCAL_MACHINE|HKEY_CURRENT_USER)\\[^\s\"']+", registry_repl, text, flags=re.IGNORECASE)
    text = re.sub(r"(?<!\\)\\\\[A-Za-z0-9_.-]+\\[A-Za-z0-9$_.-]+(?:\\[^\s\"']*)?", unc_repl, text)
    text = re.sub(r"[A-Za-z]:\\[^\s\"']+", windows_repl, text)

    important_unix = {
        "/root/.ssh/authorized_keys": "[UNIX_PATH] ssh authorized keys",
        "/etc/passwd": "[UNIX_PATH] etc passwd",
        "/etc/shadow": "[UNIX_PATH] etc shadow",
        "/var/log": "[UNIX_PATH] log directory",
        "/tmp": "[UNIX_PATH] tmp directory",
    }
    for path, repl in sorted(important_unix.items(), key=lambda x: len(x[0]), reverse=True):
        text = re.sub(re.escape(path), " " + repl + " ", text, flags=re.IGNORECASE)
    text = re.sub(r"(?<!\w)/(etc|var|tmp|root|home|usr|bin|sbin|opt|dev|proc|sys|lib|mnt)(?:/[A-Za-z0-9._-]+)*", " [UNIX_PATH] ", text)
    return text

def normalize_security_entities_dl(text):
    text = str(text)
    text = re.sub(r"https?://attack\.mitre\.org/techniques/T\d{4}(?:/\d{3})?/?", " [MITRE_REF] ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bT\d{4}(?:\.\d{3})?\b", " [MITRE_ID] ", text, flags=re.IGNORECASE)
    text = normalize_cve_dl(text)
    text = re.sub(r"https?://\S+|www\.\S+", " [URL] ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", " [EMAIL] ", text)
    text = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", " [IPV4] ", text)
    text = re.sub(r"\b(?:[A-Fa-f0-9]{1,4}:){2,7}[A-Fa-f0-9]{1,4}\b", " [IPV6] ", text)
    text = re.sub(r"\b[a-fA-F0-9]{32}\b|\b[a-fA-F0-9]{40}\b|\b[a-fA-F0-9]{64}\b", " [HASH] ", text)
    text = normalize_paths_dl(text)
    text = re.sub(r"\b[A-Za-z0-9+/]{80,}={0,2}\b", " [BASE64] ", text)
    text = re.sub(r"\bpowershell(?:\.exe)?\b", " [POWERSHELL] powershell ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bcmd(?:\.exe)?\b", " [CMD] cmd ", text, flags=re.IGNORECASE)
    return text

def normalize_text_for_deeplearning(text):
    if pd.isna(text):
        return ""
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = remove_light_boilerplate(text)
    text = add_sqli_special_signals(text)
    text = normalize_security_entities_dl(text)
    text = text.replace("`", " ")
    text = text.replace("\u201c", '"').replace("\u201d", '"').replace("\u2018", "'").replace("\u2019", "'")
    text = re.sub(r"#{1,6}|-{3,}|_{3,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return compact_spaces(text)

## 2. Load raw dataset va tao cot DL_Text

In [ ]:
raw = pd.read_csv(RAW_PATH)
required_columns = {"Cleaned_Text", "Labels"}
missing = required_columns.difference(raw.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

df = raw[["Cleaned_Text", "Labels"]].rename(columns={"Cleaned_Text": "Raw_Text"}).copy()
df["DL_Text"] = df["Raw_Text"].apply(normalize_text_for_deeplearning)
df["Cleaned_Text_OldStyle"] = df["Raw_Text"].apply(normalize_cti_text)
df["Tokenized_Text"] = df["DL_Text"].apply(lambda x: " ".join(cti_tokenizer(x)))
df["Labels"] = df["Labels"].apply(normalize_labels)
df = update_rare_labels(df, RARE_LABEL_MAPPING)

df = df[
    df["DL_Text"].str.strip().ne("")
    & df["Labels"].str.strip().ne("")
].copy()
df = df.drop_duplicates(subset=["DL_Text", "Labels"]).reset_index(drop=True)

print("Raw rows:", len(raw))
print("Rows after basic cleanup:", len(df))
display(df.head(5))

## 3. Merge duplicate text va validate label leakage

Merge theo `DL_Text`, khong theo `Tokenized_Text`, de tranh gom nham cac cau khac nhau nhung token hoa giong nhau.

In [ ]:
def merge_first(series):
    for value in series:
        if str(value).strip():
            return value
    return ""

merged = (
    df.groupby("DL_Text", as_index=False)
      .agg({
          "Labels": merge_label_strings,
          "Raw_Text": merge_first,
          "Cleaned_Text_OldStyle": merge_first,
          "Tokenized_Text": merge_first,
      })
)
merged["Labels"] = merged.apply(lambda row: manual_resolve_hard_conflict(row["DL_Text"], row["Labels"]), axis=1)
merged = update_rare_labels(merged, RARE_LABEL_MAPPING)

invalid_mask = ~merged["Labels"].apply(is_valid_label_string)
if invalid_mask.any():
    raise ValueError(merged.loc[invalid_mask, "Labels"].head(10).tolist())

leakage_mask = merged["DL_Text"].str.contains(r"\bT\d{4}(?:\.\d{3})?\b", case=False, regex=True, na=False)
if leakage_mask.any():
    raise ValueError(f"MITRE IDs remain in DL_Text rows: {int(leakage_mask.sum())}")

merged["label_count"] = merged["Labels"].apply(lambda s: len(label_set(s)))
too_many = merged["label_count"] > MAX_LABELS_AFTER_MAPPING
print("Rows after merge:", len(merged))
print("Rows with >3 labels:", int(too_many.sum()))
if too_many.any():
    display(merged.loc[too_many, ["DL_Text", "Labels", "label_count"]].head(10))

processed_dl = merged[["DL_Text", "Cleaned_Text_OldStyle", "Tokenized_Text", "Labels"]].reset_index(drop=True)
display(processed_dl.head())

## 4. Quality report

Cac chi so nay giup xem dataset moi co phu hop hon cho deep learning hay khong: do dai text, so nhan, so special tokens, va cac mau qua ngan.

In [ ]:
def explode_labels(labels_series):
    return labels_series.astype(str).str.split(",").explode().str.strip()

word_len = processed_dl["DL_Text"].str.split().str.len()
char_len = processed_dl["DL_Text"].str.len()
label_counts = explode_labels(processed_dl["Labels"]).value_counts()
cardinality = processed_dl["Labels"].apply(lambda s: len(label_set(s)))

quality_report = {
    "rows": int(len(processed_dl)),
    "num_labels": int(len(label_counts)),
    "min_label_support": int(label_counts.min()),
    "median_label_support": float(label_counts.median()),
    "max_label_support": int(label_counts.max()),
    "avg_labels_per_sample": float(cardinality.mean()),
    "short_rows_1_word": int((word_len <= 1).sum()),
    "short_rows_5_words_or_less": int((word_len <= 5).sum()),
    "long_rows_over_512_words": int((word_len > 512).sum()),
    "special_tokens": SPECIAL_TOKENS,
}

print(json.dumps({k: v for k, v in quality_report.items() if k != "special_tokens"}, indent=2))
display(pd.DataFrame({
    "metric": ["word_len", "char_len", "label_cardinality"],
    "p01": [word_len.quantile(0.01), char_len.quantile(0.01), cardinality.quantile(0.01)],
    "p05": [word_len.quantile(0.05), char_len.quantile(0.05), cardinality.quantile(0.05)],
    "p50": [word_len.quantile(0.50), char_len.quantile(0.50), cardinality.quantile(0.50)],
    "p95": [word_len.quantile(0.95), char_len.quantile(0.95), cardinality.quantile(0.95)],
    "p99": [word_len.quantile(0.99), char_len.quantile(0.99), cardinality.quantile(0.99)],
    "max": [word_len.max(), char_len.max(), cardinality.max()],
}))
display(label_counts.head(20).rename_axis("label").reset_index(name="sample_count"))
display(processed_dl.assign(word_len=word_len).sort_values("word_len").head(15))

## 5. Split Stage 1 / Stage 2 va ghi file

Stage 1 chi giu frequent labels theo threshold 30 nhu pipeline cu. Stage 2 giu cac samples co it nhat mot rare label de dung retrieval/vector DB/human review.

In [ ]:
stage1_dl, stage2_dl, label_frequency = split_stage_datasets(processed_dl, threshold=STAGE1_THRESHOLD)

processed_path = OUTPUT_DIR / "attack_dataset_dl_processed.csv"
stage1_path = OUTPUT_DIR / "attack_dataset_dl_stage1_frequent.csv"
stage2_path = OUTPUT_DIR / "attack_dataset_dl_stage2_rare.csv"
label_freq_path = OUTPUT_DIR / "attack_dataset_dl_label_frequency.csv"
special_tokens_path = OUTPUT_DIR / "special_tokens.json"
report_path = OUTPUT_DIR / "preprocessing_dl_report.json"

processed_dl.to_csv(processed_path, index=False, encoding="utf-8")
stage1_dl.to_csv(stage1_path, index=False, encoding="utf-8")
stage2_dl.to_csv(stage2_path, index=False, encoding="utf-8")
label_frequency.rename_axis("label").reset_index(name="sample_count").to_csv(label_freq_path, index=False, encoding="utf-8")

with open(special_tokens_path, "w", encoding="utf-8") as f:
    json.dump({"additional_special_tokens": SPECIAL_TOKENS}, f, indent=2, ensure_ascii=False)

summary = {
    **quality_report,
    "raw_rows": int(len(raw)),
    "stage1_threshold": int(STAGE1_THRESHOLD),
    "stage1_rows": int(len(stage1_dl)),
    "stage1_labels": int((label_frequency >= STAGE1_THRESHOLD).sum()),
    "stage2_rows": int(len(stage2_dl)),
    "stage2_rare_labels": int((label_frequency < STAGE1_THRESHOLD).sum()),
    "processed_path": str(processed_path),
    "stage1_path": str(stage1_path),
    "stage2_path": str(stage2_path),
    "label_frequency_path": str(label_freq_path),
    "special_tokens_path": str(special_tokens_path),
}
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))

## 6. Cach dung trong notebook train Transformer

Trong notebook train, doi data path va text column:

```python
DATA_PATH = PROJECT_ROOT / "dataset" / "processed_deeplearning" / "attack_dataset_dl_stage1_frequent.csv"
TEXT_COL = "DL_Text"
```

Neu model/tokenizer cho phep them token, add special tokens truoc khi tao model:

```python
with open(PROJECT_ROOT / "dataset" / "processed_deeplearning" / "special_tokens.json", encoding="utf-8") as f:
    special_tokens = json.load(f)

num_added = tokenizer.add_special_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))
```

Voi `cisco-ai/SecureBERT2.0-base`, nen bat dau voi `MAX_LENGTH = 1024`. Neu GPU yeu, thu `MAX_LENGTH = 512` hoac chunking head+tail.